In [1]:
from __future__ import annotations

import re
import zipfile
from pathlib import Path
import numpy as np

import pandas as pd


# ----------------------------
# 1) Set your folders
# ----------------------------
RAW_DIR = Path(r"C:\Users\asinha\Coding_KAI\_Own_project\Congestion Regime Prediction\Data\a_raw_download\data_speed_tt")
YEARS_TO_PROCESS = [2023, 2024, 2025]          # or [2021, 2022, 2023, 2024, 2025]


OUT_DIR = Path(r"..\..\Data\b_processed_data\ritis_speed_tt_5min")
OUT_DIR.mkdir(parents=True, exist_ok=True)



# SAVE_CSV_TOO = True  # set True if you also want CSV output

In [2]:
def select_zip_files_by_year(raw_dir: Path, years: list[int]) -> list[Path]:
    """
    Select zip files where the filename ends with the year, like *_2021.zip
    """
    years_str = {str(y) for y in years}
    zips = []
    for zp in sorted(raw_dir.glob("*.zip")):
        stem = zp.stem  # filename without .zip
        year = stem[-4:] if len(stem) >= 4 else ""
        if year in years_str:
            zips.append(zp)
    return zips


def read_csv_from_zip(zip_path: Path, contains: str) -> pd.DataFrame:
    """
    Reads the first CSV inside the zip whose name contains `contains` (case-insensitive).
    Example: contains="tmc_identification" or contains="speed_tt_2021"
    """
    contains = contains.lower()

    with zipfile.ZipFile(zip_path, "r") as z:
        members = [m for m in z.namelist() if m.lower().endswith(".csv")]
        if not members:
            raise ValueError(f"No CSV files found inside {zip_path.name}")

        match = [m for m in members if contains in m.lower()]
        if not match:
            raise ValueError(f"No CSV matched '{contains}' inside {zip_path.name}. Found: {members}")

        with z.open(match[0]) as f:
            return pd.read_csv(f, low_memory=False)


# ----------------------------
# 3) Processing
# ----------------------------
def process_speed_tt(df: pd.DataFrame) -> pd.DataFrame:
    # Rename columns (based on your sample)
    df = df.rename(columns={
        "tmc_code": "tmc",
        "measurement_tstamp": "ts_local",
        "speed": "speed_mph",
        "historical_average_speed": "hist_avg_speed_mph",
        "reference_speed": "ref_speed_mph",
        "travel_time_seconds": "travel_time_sec",
        "confidence_score": "confidence",
        "cvalue": "cvalue",
    }).copy()

    # Parse local timestamp and convert to UTC
    df["ts_local"] = pd.to_datetime(df["ts_local"], errors="coerce")

    df["ts_local"] = df["ts_local"].dt.tz_localize(
        "America/New_York",
        ambiguous="NaT",          # <- changed
        nonexistent="shift_forward"
    )

    df = df.dropna(subset=["ts_local"])  # <- add this line
    df["ts_utc"] = df["ts_local"].dt.tz_convert("UTC")

    # Numeric safety
    for c in ["speed_mph", "ref_speed_mph", "travel_time_sec", "confidence", "cvalue", "hist_avg_speed_mph"]:
        if c in df.columns:
            df[c] = pd.to_numeric(df[c], errors="coerce")

    # Travel time in minutes
    df["travel_time_min"] = df["travel_time_sec"] / 60.0

    # TTI using reference speed: tti = ref_speed / speed
    df["tti"] = np.where(
        (df["speed_mph"] > 0) & (df["ref_speed_mph"] > 0),
        df["ref_speed_mph"] / df["speed_mph"],
        np.nan
    )

    df = df.dropna(subset=["tmc", "ts_utc"])
    df = df.sort_values(["tmc", "ts_utc"]).reset_index(drop=True)
    return df


def process_tmc_lookup(lu: pd.DataFrame) -> pd.DataFrame:
    # Ensure key column is named 'tmc'
    if "tmc_code" in lu.columns and "tmc" not in lu.columns:
        lu = lu.rename(columns={"tmc_code": "tmc"}).copy()

    # Optional: keep only I-66 rows if present
    if "road" in lu.columns:
        lu = lu[lu["road"].astype(str).str.upper().str.contains("I-66", na=False)].copy()

    # Parse dates so we can pick a consistent row
    if "active_start_date" in lu.columns:
        lu["active_start_date"] = pd.to_datetime(lu["active_start_date"], errors="coerce")

    # Keep ONE row per tmc. Choose the latest active_start_date.
    lu = lu.sort_values(["tmc", "active_start_date"]).drop_duplicates(subset=["tmc"], keep="last")

    return lu


def main() -> None:
    
    zip_files = select_zip_files_by_year(RAW_DIR, YEARS_TO_PROCESS)
    if not zip_files:
        raise FileNotFoundError(f"No zip files found for years={YEARS_TO_PROCESS} in {RAW_DIR}")

    # Read both CSVs from the same zip
    # all_years = []

    for zp in zip_files:
        year = int(zp.stem[-4:])
        print(f"\nProcessing year {year}: {zp.name}")

        # Read both CSVs from this zip
        speed_tt = read_csv_from_zip(zp, contains=str(year))
        tmc_lu = read_csv_from_zip(zp, contains="tmc_identification")

        # Process
        df = process_speed_tt(speed_tt)
        lu = process_tmc_lookup(tmc_lu)

        # Merge
        out = df.merge(lu, on="tmc", how="left", validate="m:1")
        out["year"] = year

        # Save per-year
        out_parquet = OUT_DIR / f"I-66_Research_Speed_TT_{year}_silver.parquet"
        out.to_parquet(out_parquet, index=False)

        out_csv = OUT_DIR / f"I-66_Research_Speed_TT_{year}_silver.csv"
        out.to_csv(out_csv, index=False)

        print(out.head(10))
        print(f"Saved parquet: {out_parquet}")
        print(f"Saved csv: {out_csv}")
        print(f"Rows: {len(out):,} | Cols: {len(out.columns)}")

        # all_years.append(out)

    ##### If we want to combine, Append across years at the end
    # combined = pd.concat(all_years, ignore_index=True)

    # combined_years = f"{min(YEARS_TO_PROCESS)}_{max(YEARS_TO_PROCESS)}"
    # combined_parquet = OUT_DIR / f"I-66_Research_Speed_TT_{combined_years}_silver.parquet"
    # combined.to_parquet(combined_parquet, index=False)

    # combined_csv = OUT_DIR / f"I-66_Research_Speed_TT_{combined_years}_silver.csv"
    # combined.to_csv(combined_csv, index=False)

    # print(f"\nSaved combined parquet: {combined_parquet}")
    # print(f"Saved combined csv: {combined_csv}")
    # print("Done.")

if __name__ == "__main__":
    main()


Processing year 2023: I-66_Research_Speed_TT_2023.zip
         tmc                  ts_local  speed_mph  hist_avg_speed_mph  \
0  110+04164 2023-01-01 00:00:00-05:00      54.20                50.0   
1  110+04164 2023-01-01 00:05:00-05:00      54.15                50.0   
2  110+04164 2023-01-01 00:10:00-05:00      50.60                50.0   
3  110+04164 2023-01-01 00:15:00-05:00      51.54                50.0   
4  110+04164 2023-01-01 00:20:00-05:00      52.00                50.0   
5  110+04164 2023-01-01 00:25:00-05:00      50.43                50.0   
6  110+04164 2023-01-01 00:30:00-05:00      43.15                50.0   
7  110+04164 2023-01-01 00:35:00-05:00      46.45                50.0   
8  110+04164 2023-01-01 00:40:00-05:00      49.18                50.0   
9  110+04164 2023-01-01 00:45:00-05:00      48.39                50.0   

   ref_speed_mph  travel_time_sec  confidence  cvalue  \
0           50.0             7.16        30.0   100.0   
1           50.0           